In [9]:
%load_ext autoreload
%autoreload 2

import scanpy as sc
import sys
import os
import anndata
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import torch
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

print("CUDA Available: ", torch.cuda.is_available())
print("Number of GPUs: ", torch.cuda.device_count())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
CUDA Available:  True
Number of GPUs:  1


In [ ]:
data = sc.read_h5ad("../../../original_data/plant_all_days.h5ad")

In [11]:
print(data)
group_map = {'Nodule (21 dpi)': 21, 'Nodule (12 dpi)': 12, 'Root': 0}
data.obs['Group'] = data.obs['Sample_time'].map(group_map)
# print(data.obs[['Sample_time', 'Group']].drop_duplicates())

AnnData object with n_obs × n_vars = 26712 × 39337
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'SNF', 'means_ForPickMock'
    uns: 'Cell type_colors'
    obsm: 'X_scvi', 'X_umap'
    layers: 'empty', 'normalize_log', 'raw'
    obsp: 'connectivities', 'distances'


In [12]:
# See unique time points
print(data.obs['Sample_time'].unique())

# See value counts
print(data.obs['Sample_time'].unique())

print(data.obs['Cell type'].unique())
print(data)
# make train data
plant_data_12 = data[data.obs['Sample_time'] == 'Nodule (12 dpi)'].copy()
plant_data_21 = data[data.obs['Sample_time'] == 'Nodule (21 dpi)'].copy()
plant_data_0 = data[data.obs['Sample_time'] == 'Root'].copy() 
print(plant_data_12.obs['Sample_time'].unique())
print(plant_data_21.obs['Sample_time'].unique())
print(plant_data_0.obs['Sample_time'].unique())

['Nodule (21 dpi)', 'Nodule (12 dpi)', 'Root']
Categories (3, object): ['Nodule (21 dpi)', 'Nodule (12 dpi)', 'Root']
['Nodule (21 dpi)', 'Nodule (12 dpi)', 'Root']
Categories (3, object): ['Nodule (21 dpi)', 'Nodule (12 dpi)', 'Root']
['2: Outer cortex', '12: Infected cells', '9: Vascular bundle', '4: Outer cortex', '0: Uninfected cells', ..., '6: Unknown', '10: Unknown', '3: Vascular bundle', '13: Unknown', '14: Unknown']
Length: 15
Categories (15, object): ['0: Uninfected cells', '1: Inner cortex', '2: Outer cortex', '3: Vascular bundle', ..., '11: Uninfected cells', '12: Infected cells', '13: Unknown', '14: Unknown']
AnnData object with n_obs × n_vars = 26712 × 39337
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time', 'Group'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'SNF', 'means_ForPickMock'
    uns: 'Cell type_colors'
    obsm: 'X_scvi', 'X_umap'

In [13]:
# Shuffle the sampled data indices
total_indices = np.arange(plant_data_12.shape[0])
#print(total_indices)
np.random.shuffle(total_indices)
#print(total_indices)

# Split into train, validation, and test datasets
train_12_size = int(0.8 * plant_data_12.shape[0])

train_12_indices = total_indices[:train_12_size]
#print(train_12_indices)
test_12_indices = total_indices[train_12_size:]

train_12 = plant_data_12[train_12_indices, :]
test_12 = plant_data_12[test_12_indices, :]
train_12.write_h5ad('results/plant_experiment1/plant_paper_train_day12_adata.h5ad')
test_12.write_h5ad('results/plant_experiment1/plant_paper_test_day12_adata.h5ad')

In [14]:
# Shuffle the sampled data indices
total_indices = np.arange(plant_data_0.shape[0])
#print(total_indices)
np.random.shuffle(total_indices)
#print(total_indices)

# Split into train, validation, and test datasets
train_0_size = int(0.8 * plant_data_0.shape[0])

train_0_indices = total_indices[:train_0_size]
#print(train_0_indices)
test_0_indices = total_indices[train_0_size:]

train_0 = plant_data_0[train_0_indices, :]
test_0 = plant_data_0[test_0_indices, :]
train_0.write_h5ad('results/plant_experiment1/plant_paper_train_day0_adata.h5ad')
test_0.write_h5ad('results/plant_experiment1/plant_paper_test_day0_adata.h5ad')

In [15]:
# Shuffle the sampled data indices
total_indices = np.arange(plant_data_21.shape[0])
#print(total_indices)
np.random.shuffle(total_indices)
#print(total_indices)

# Split into train, validation, and test datasets
train_21_size = int(0.8 * plant_data_21.shape[0])

train_21_indices = total_indices[:train_21_size]
#print(train_21_indices)
test_21_indices = total_indices[train_21_size:]

train_21 = plant_data_21[train_21_indices, :]
test_21 = plant_data_21[test_21_indices, :]
train_21.write_h5ad('results/plant_experiment1/plant_paper_train_day21_adata.h5ad')
test_21.write_h5ad('results/plant_experiment1/plant_paper_test_day21_adata.h5ad')

In [18]:
train_data_part1 = sc.read_h5ad('results/plant_experiment1/plant_paper_train_day12_adata.h5ad')                                                                                    
print(train_data_part1.obs['Group'].unique())
train_data_part2 = sc.read_h5ad('results/plant_experiment1/plant_paper_train_day21_adata.h5ad')
print(train_data_part2.obs['Group'].unique())
combined = anndata.concat([train_data_part1, train_data_part2])
print(combined.obs['Group'].unique())
combined.write_h5ad('results/plant_experiment1/plant_experiment1_train_adata.h5ad')

[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]


In [ ]:
data = sc.read_h5ad('../input/plant_experiment1_train_adata.h5ad')
print(data.obs['Group'].unique())

[12, 21]
Categories (2, int64): [12, 21]


In [20]:
data = sc.read_h5ad('results/plant_experiment1/plant_paper_test_day21_adata.h5ad')
print(data)
print(data.obs['Group'].unique())

AnnData object with n_obs × n_vars = 2401 × 39337
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time', 'Group'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'SNF', 'means_ForPickMock'
    uns: 'Cell type_colors'
    obsm: 'X_scvi', 'X_umap'
    layers: 'empty', 'normalize_log', 'raw'
    obsp: 'connectivities', 'distances'
[21]
Categories (1, int64): [21]


In [ ]:
print(data.obs)

                                       n_genes  n_counts Cluster leiden_R  \
ATGCATGAGTGGCGAT-1-batch-nodule_small      664     785.0       9        9   
TCAGGGCCATTCTTCA-1-batch-nodule_small     1893    2969.0       0        0   
ATGGGAGTCGATACTG-1-batch-nodule_small     2144    3104.0       2        2   
TACAGGTTCACAGTGT-1-batch-nodule_small     1254    1631.0       5        5   
ATCCGTCGTATCACCA-1-batch-nodule_small     1801    2863.0       4        4   
...                                        ...       ...     ...      ...   
CTACATTGTGGCATCC-1-batch-nodule_large     1149    1555.0       0        0   
ACTATCTTCAGTCCGG-1-batch-nodule_large     1248    1581.0       1        1   
ACGGTCGAGCTAAACA-1-batch-nodule_large      989    1211.0       1        1   
GAGCCTGCAGCGTTGC-1-batch-nodule_large      886    1129.0       0        0   
AATGGCTCACACTTAG-1-batch-nodule_large      907    1079.0       2        2   

                                                 Cell type      Sample_time